# Analyse de sentiments sur des tweets

Dans ce TP, nous allons classer des tweets en trois catégories : **négatif**, **neutre** ou **positif**.

Nous utiliserons le jeu de données [TweetEval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) (tâche *sentiment*) : environ 60 000 tweets en anglais, annotés à la main.

Au programme :

1. préparer des batchs sans padding grâce aux `PackedSequence` de PyTorch ;
2. entraîner un LSTM bidirectionnel avec PyTorch Lightning, avec arrêt anticipé (*early stopping*) et sauvegarde du meilleur modèle ;
3. comparer le résultat à une référence TF-IDF et à des transformeurs pré-entraînés : un encodeur que l'on affine soi-même, et un modèle déjà affiné sur cette tâche.

*Pensez à activer le GPU : Exécution > Modifier le type d'exécution > GPU.*

## Imports

In [ ]:
!pip install -q datasets lightning torchmetrics

In [ ]:
import collections

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import spacy
import torch
from datasets import load_dataset
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, recall_score
from torch import nn
from torch.nn.utils.rnn import PackedSequence, pack_sequence
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import MulticlassRecall

lightning.seed_everything(0)

## Chargement des données

La bibliothèque `datasets` de Hugging Face télécharge le jeu de données déjà découpé en entraînement, validation et test.

In [ ]:
tweets = load_dataset("cardiffnlp/tweet_eval", "sentiment")
label_names = ["négatif", "neutre", "positif"]
print(tweets)

La cellule suivante affiche dix tweets d'entraînement avec leur label, la répartition des classes et la longueur des tweets (en mots).

In [ ]:
train_df = tweets["train"].to_pandas()
train_df["sentiment"] = train_df.label.map(dict(enumerate(label_names)))
print(train_df.sample(10, random_state=0)[["sentiment", "text"]].to_string())

train_df.sentiment.value_counts().plot.bar(title="Répartition des classes")
plt.show()

train_df.text.str.split().str.len().plot.hist(bins=30, title="Longueur des tweets (mots)")
plt.show()

*Comment ont été traités les noms d'utilisateurs ? Les classes sont-elles équilibrées, et quelle conséquence pour le choix de la métrique ? Quelle est la longueur typique d'un tweet ?*

### Solution

- Les noms d'utilisateurs sont remplacés par `@user` : les tweets sont anonymisés.
- La classe *neutre* est la plus fréquente, la classe *négatif* la plus rare : la justesse (*accuracy*) favoriserait un modèle qui prédit surtout *neutre*. La métrique officielle de TweetEval est le **rappel moyen par classe** (*macro recall*), qui donne le même poids à chaque classe.
- Un tweet fait une vingtaine de mots, rarement plus de 40.

## Référence : TF-IDF et régression logistique

Avant d'entraîner un réseau de neurones, mesurons ce qu'obtient un modèle simple : une régression logistique sur une représentation TF-IDF des tweets. Si le réseau ne fait pas mieux, il est inutile !

In [ ]:
vectorizer = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(tweets["train"]["text"])
X_val_tfidf = vectorizer.transform(tweets["validation"]["text"])

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_tfidf, tweets["train"]["label"])
baseline_recall = recall_score(tweets["validation"]["label"],
                               baseline.predict(X_val_tfidf), average="macro")
print(f"Rappel moyen TF-IDF : {baseline_recall:.3f}")

## Tokenisation avec spaCy

Pour un réseau récurrent, chaque tweet doit devenir une séquence d'indices de mots. Nous utilisons le tokeniseur de spaCy, qui gère la ponctuation, les émoticônes et les contractions bien mieux qu'un simple `split()`.

`spacy.blank("en")` crée un pipeline anglais vide : il ne contient que le tokeniseur, ce qui est très rapide.

In [ ]:
nlp = spacy.blank("en")


def tokenize(text: str) -> list[str]:
  return [token.lower_ for token in nlp.tokenizer(text) if not token.is_space]


print(tokenize(tweets["train"][0]["text"]))

## Construction du vocabulaire

Le vocabulaire est construit à partir de l'ensemble d'entraînement **seulement**, avec les mots vus au moins 2 fois ; l'indice `0` est réservé aux mots inconnus (`<unk>`). `numericalize` transforme un tweet en liste d'indices.

In [ ]:
counts = collections.Counter(
    word for text in tweets["train"]["text"] for word in tokenize(text))
vocab = {"<unk>": 0}
for word, count in counts.most_common():
  if count >= 2:
    vocab[word] = len(vocab)
print(f"Taille du vocabulaire : {len(vocab)}")


def numericalize(text: str) -> list[int]:
  return [vocab.get(word, vocab["<unk>"]) for word in tokenize(text)]


print(numericalize(tweets["train"][0]["text"]))

*Pourquoi ne pas utiliser les ensembles de validation et de test pour construire le vocabulaire ?*

### Solution

Le vocabulaire fait partie du modèle : le construire avec les données de validation ou de test ferait « fuiter » de l'information sur ces données, et l'évaluation serait trop optimiste.

## Des batchs sans padding : `PackedSequence`

Les tweets n'ont pas tous la même longueur. Plutôt que de compléter les séquences d'un batch avec du padding, PyTorch peut les **empaqueter** dans une `PackedSequence` : tous les tokens du batch sont rangés bout à bout, pas de temps par pas de temps, sans aucun padding.

- `data` : d'abord le premier token de chaque séquence, puis le deuxième token de chaque séquence qui en a au moins deux, etc. ;
- `batch_sizes` : le nombre de séquences encore en cours à chaque pas de temps ;
- `sorted_indices` / `unsorted_indices` : l'ordre (par longueur décroissante) dans lequel les séquences sont rangées, et de quoi revenir à l'ordre d'origine.

Les couches récurrentes de PyTorch (`nn.RNN`, `nn.GRU`, `nn.LSTM`) acceptent directement une `PackedSequence` : à chaque pas de temps, elles ne calculent que sur les séquences encore en cours, et leur dernier état caché `h_n` est celui du dernier *vrai* token de chaque séquence, remis dans l'ordre d'origine.

*Exécutez la cellule suivante et retrouvez comment `data` et `batch_sizes` sont construits à partir des trois séquences.*

In [ ]:
example = pack_sequence([torch.tensor([1, 2, 3]), torch.tensor([4, 5]),
                         torch.tensor([6, 7, 8, 9])], enforce_sorted=False)
print(example)

## `Dataset` et `DataLoader`

`TweetDataset` renvoie chaque tweet sous forme de tenseur d'indices, avec son label.

In [ ]:
class TweetDataset(Dataset):
  def __init__(self, split) -> None:
    # Un tweet sans aucun token devient un unique <unk> : une séquence vide
    # ne peut pas être empaquetée
    self.sequences = [torch.tensor(numericalize(text) or [vocab["<unk>"]])
                      for text in split["text"]]
    self.labels = split["label"]

  def __len__(self) -> int:
    return len(self.labels)

  def __getitem__(self, i: int) -> tuple[torch.Tensor, int]:
    return self.sequences[i], self.labels[i]


train_dataset = TweetDataset(tweets["train"])
val_dataset = TweetDataset(tweets["validation"])
test_dataset = TweetDataset(tweets["test"])
print(train_dataset[0])

*Codez la fonction `collate(batch)` que les `DataLoader` utiliseront pour former leurs batchs : elle reçoit une liste de couples `(tenseur d'indices, label)` et renvoie `(séquences empaquetées, labels)`. Utilisez `pack_sequence(..., enforce_sorted=False)` : les séquences n'ont pas à être triées par longueur, `pack_sequence` s'en charge.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
def collate(batch: list[tuple[torch.Tensor, int]]) -> tuple[PackedSequence, torch.Tensor]:
  sequences, labels = zip(*batch)
  return pack_sequence(sequences, enforce_sorted=False), torch.tensor(labels)

### Des batchs empaquetés

Les `DataLoader` forment leurs batchs avec `collate`. Comparons, sur un batch d'entraînement, le nombre de tokens empaquetés à la taille qu'aurait le même batch complété avec du padding :

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          collate_fn=collate)
val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate)
test_loader = DataLoader(test_dataset, batch_size=64, collate_fn=collate)

packed, labels = next(iter(train_loader))
# Nombre de tweets × longueur du plus long
padded_size = len(labels) * len(packed.batch_sizes)
print(f"Tokens du batch : {packed.data.numel()} empaquetés contre {padded_size} "
      f"avec padding, soit {1 - packed.data.numel() / padded_size:.0%} de padding évité")

Les longueurs des tweets d'un batch varient beaucoup : complété avec du padding, un batch contiendrait environ deux fois plus de positions que de vrais tokens, et le LSTM ferait autant de calculs inutiles. La `PackedSequence` les évite, et le modèle n'a plus besoin d'indice de padding.

## Le modèle : un LSTM bidirectionnel

*Complétez le modèle suivant, qui prend en entrée une `PackedSequence` d'indices de mots : des embeddings, un LSTM bidirectionnel, et une couche de sortie qui produit un logit par classe à partir des derniers états cachés des deux directions. Ajoutez un dropout de 0,3 sur les embeddings et avant la couche de sortie.*

*`nn.Embedding` et `nn.Dropout` n'acceptent pas de `PackedSequence` : appliquez-les aux tokens `packed.data`, puis reconstruisez une `PackedSequence` avec `packed._replace(data=...)`, qui garde `batch_sizes` et les indices de tri.*

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    # Votre code ici

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    # Votre code ici
    pass

### Solution

In [ ]:
class SentimentLSTM(nn.Module):
  def __init__(self, vocab_size: int, embedding_dim: int = 128,
               hidden_size: int = 128, n_classes: int = 3) -> None:
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, hidden_size, bidirectional=True)
    self.dropout = nn.Dropout(0.3)
    self.head = nn.Linear(2 * hidden_size, n_classes)

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    # Embeddings et dropout de tous les tokens du batch à la fois, remis dans
    # une PackedSequence de même structure
    embedded = packed._replace(data=self.dropout(self.embedding(packed.data)))
    _, (h_n, _) = self.lstm(embedded)
    # Derniers états cachés des directions avant et arrière, dans l'ordre
    # d'origine des tweets
    summary = torch.cat([h_n[-2], h_n[-1]], dim=1)
    return self.head(self.dropout(summary))


lstm = SentimentLSTM(len(vocab))
print(lstm)
print(sum(p.numel() for p in lstm.parameters()), "paramètres")

Pourquoi `h_n[-2]` et `h_n[-1]` : ce que contient `h_n` à la fin d’un GRU ou d’un LSTM, à plusieurs couches et bidirectionnel, en vidéo : [`h_n` couche par couche](https://shuuchuu.github.io/videos/nn/gru-final-hidden-state.mp4).

## Entraînement

Le `LightningModule` ci-dessous décrit la fonction de perte, la métrique (le rappel moyen par classe) et l'optimiseur. Les batchs sont des couples `(PackedSequence, labels)` : le `Trainer` les déplace lui-même sur le GPU, `PackedSequence` comprise.

In [ ]:
class SentimentClassifier(lightning.LightningModule):
  """Enveloppe Lightning : classe le sentiment de tweets empaquetés."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-3,
               n_classes: int = 3) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # torchmetrics demande une instance de métrique par étape
    self.recalls = nn.ModuleDict({
        f"{stage}_recall": MulticlassRecall(num_classes=n_classes, average="macro")
        for stage in ("val", "test")})

  def forward(self, packed: PackedSequence) -> torch.Tensor:
    return self.model(packed)

  def _step(self, batch: tuple[PackedSequence, torch.Tensor],
            stage: str) -> torch.Tensor:
    packed, labels = batch
    logits = self(packed)
    loss = nn.functional.cross_entropy(logits, labels)
    # Sans batch_size, Lightning le déduirait de packed.data : le nombre de tokens
    self.log(f"{stage}_loss", loss, batch_size=len(labels), on_step=False,
             on_epoch=True, prog_bar=True)
    if stage != "train":
      recall = self.recalls[f"{stage}_recall"]
      recall(logits, labels)
      self.log(f"{stage}_recall", recall, batch_size=len(labels), on_step=False,
               on_epoch=True, prog_bar=True)
    return loss

  def training_step(self, batch: tuple[PackedSequence, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[PackedSequence, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[PackedSequence, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: tuple[PackedSequence, torch.Tensor],
                   batch_index: int) -> torch.Tensor:
    packed, _ = batch
    return self(packed).argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)


def plot_metrics(trainer: lightning.Trainer) -> None:
  """Trace les pertes et le rappel moyen de validation au fil des epochs."""
  # Les métriques d'entraînement et de validation ne sont pas écrites sur les
  # mêmes lignes du fichier CSV : on les regroupe par epoch
  metrics = pandas.read_csv(f"{trainer.logger.log_dir}/metrics.csv").groupby(
      "epoch").mean(numeric_only=True).dropna(subset=["train_loss", "val_loss"])
  metrics[["train_loss", "val_loss"]].plot(title="Perte", marker="o")
  metrics[["val_recall"]].plot(title="Rappel moyen en validation", marker="o", legend=False)
  plt.show()

*Entraînez le modèle pendant 10 epochs au plus, avec deux callbacks :*

- *`EarlyStopping`, qui arrête l'entraînement quand `val_recall` ne progresse plus pendant 2 epochs ;*
- *`ModelCheckpoint`, qui garde les poids du meilleur modèle selon `val_recall`.*

*Tracez ensuite les courbes d'apprentissage avec `plot_metrics`, qui lit les métriques d'un `CSVLogger`.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
model = SentimentClassifier(SentimentLSTM(len(vocab)))
checkpoint = ModelCheckpoint(monitor="val_recall", mode="max")
trainer = lightning.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name="sentiment_lstm"),
    callbacks=[EarlyStopping(monitor="val_recall", mode="max", patience=2),
               checkpoint])
trainer.fit(model, train_loader, val_loader)
print(f"Meilleur rappel moyen en validation : {checkpoint.best_model_score:.3f}")
plot_metrics(trainer)

## Évaluation finale sur l'ensemble de test

*Évaluez le meilleur modèle sur l'ensemble de test : son rappel moyen, puis le rapport de classification et la matrice de confusion de ses prédictions, avec la fonction `show_evaluation` ci-dessous. Faites de même pour la référence TF-IDF.*

*Quelles classes sont le plus souvent confondues ? Le LSTM fait-il mieux que la référence ?*

In [ ]:
def show_evaluation(truths: numpy.ndarray, predictions: numpy.ndarray,
                    title: str) -> None:
  """Affiche le rapport de classification et la matrice de confusion."""
  print(title)
  print(classification_report(truths, predictions, target_names=label_names,
                              digits=3))
  ConfusionMatrixDisplay.from_predictions(truths, predictions,
                                          display_labels=label_names,
                                          normalize="true")
  plt.title(title)
  plt.show()

In [ ]:
# Votre code ici

### Solution

In [ ]:
trainer.test(model, test_loader, ckpt_path="best")

# trainer.predict renvoie la liste des sorties de predict_step, une par batch
predictions = torch.cat(trainer.predict(model, test_loader, ckpt_path="best")).numpy()
truths = numpy.array(tweets["test"]["label"])
show_evaluation(truths, predictions, "LSTM bidirectionnel")

baseline_predictions = baseline.predict(vectorizer.transform(tweets["test"]["text"]))
show_evaluation(truths, baseline_predictions, "Référence TF-IDF")

- Les confusions se font surtout avec la classe *neutre* ; *négatif* et *positif* sont rarement confondus entre eux.
- Le LSTM appris depuis zéro ne fait qu'un peu mieux que la référence TF-IDF (rappel moyen d'environ 0,59 contre 0,56 en test) : avec quelques dizaines de milliers de tweets seulement, il manque de données pour apprendre le sens des mots. C'est la motivation des modèles pré-entraînés.

## Pour comparer : un transformeur pré-entraîné

Le modèle [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) est un RoBERTa pré-entraîné sur des millions de tweets, puis affiné sur cette même tâche. La fonction `pipeline` de `transformers` permet de l'utiliser en une ligne.

*Évaluez-le sur les 1 000 premiers tweets de test et comparez son rappel moyen à celui du LSTM sur ces mêmes tweets. Qu'en concluez-vous ?*

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                      device=0 if torch.cuda.is_available() else -1)
print(classifier("I love this course on PyTorch!"))

In [ ]:
# Votre code ici

### Solution

In [ ]:
subset = tweets["test"].select(range(1000))
name_to_label = {"negative": 0, "neutral": 1, "positive": 2}
roberta_predictions = [name_to_label[output["label"]]
                       for output in classifier(list(subset["text"]), batch_size=64)]
print(f"RoBERTa pré-entraîné : "
      f"{recall_score(list(subset['label']), roberta_predictions, average='macro'):.3f}")
print(f"LSTM : {recall_score(truths[:1000], predictions[:1000], average='macro'):.3f}")

Le transformeur pré-entraîné fait nettement mieux (environ 0,71 contre 0,59) : il a appris le langage des tweets sur des millions d'exemples avant d'être affiné. Pour une tâche de classification de texte courante, partir d'un modèle pré-entraîné est aujourd'hui le premier réflexe ; entraîner un LSTM depuis zéro reste utile quand les contraintes de calcul sont fortes ou quand aucun modèle pré-entraîné n'existe pour la langue ou le domaine.

## Affiner soi-même un encodeur pré-entraîné

Le RoBERTa de cardiffnlp a été affiné par d'autres sur cette tâche. Faisons-le nous-mêmes, à partir d'un encodeur généraliste qui n'a jamais vu ces labels : [`distilroberta-base`](https://huggingface.co/distilbert/distilroberta-base), une version réduite de RoBERTa (82 M de paramètres). `AutoModelForSequenceClassification` lui ajoute une tête de classification à 3 sorties (voir la slide « Affiner un encodeur pré-entraîné »), et tout le modèle est entraîné avec un petit learning rate.

La cellule suivante tokenise les tweets avec le tokenizer de l'encodeur et définit l'enveloppe Lightning.

*Entraînez ce modèle pendant 2 epochs en gardant le meilleur selon `val_recall`, évaluez-le sur l'ensemble de test avec `show_evaluation`, puis comparez son rappel moyen sur les 1 000 premiers tweets de test à ceux du LSTM et du RoBERTa de cardiffnlp.*

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding)

ENCODER = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
# Tokenisation de chaque tweet : identifiants des sous-mots et masque d'attention
encoded = tweets.map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=128),
    batched=True).remove_columns(["text"]).rename_column("label", "labels")
# Le padding est fait batch par batch, à la longueur du plus long tweet du batch
collate = DataCollatorWithPadding(tokenizer)
encoder_loaders = {
    split: DataLoader(encoded[split], batch_size=32, shuffle=split == "train",
                      collate_fn=collate)
    for split in ("train", "validation", "test")}


class TransformerClassifier(lightning.LightningModule):
  """Encodeur pré-entraîné surmonté d'une nouvelle tête de classification."""

  def __init__(self, name: str, learning_rate: float = 2e-5,
               n_classes: int = 3) -> None:
    super().__init__()
    self.save_hyperparameters()
    # La tête de classification est créée avec des poids aléatoires
    self.model = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=n_classes)
    self.recalls = nn.ModuleDict({
        f"{stage}_recall": MulticlassRecall(num_classes=n_classes, average="macro")
        for stage in ("val", "test")})

  def _step(self, batch: dict[str, torch.Tensor], stage: str) -> torch.Tensor:
    # Avec batch["labels"], le modèle calcule lui-même l'entropie croisée
    outputs = self.model(**batch)
    batch_size = len(batch["labels"])
    self.log(f"{stage}_loss", outputs.loss, batch_size=batch_size, on_step=False,
             on_epoch=True, prog_bar=True)
    if stage != "train":
      recall = self.recalls[f"{stage}_recall"]
      recall(outputs.logits, batch["labels"])
      self.log(f"{stage}_recall", recall, batch_size=batch_size, on_step=False,
               on_epoch=True, prog_bar=True)
    return outputs.loss

  def training_step(self, batch: dict[str, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: dict[str, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: dict[str, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: dict[str, torch.Tensor],
                   batch_index: int) -> torch.Tensor:
    return self.model(**batch).logits.argmax(dim=1)

  def configure_optimizers(self) -> torch.optim.Optimizer:
    # Tout le modèle est affiné : un petit learning rate préserve ce que
    # l'encodeur a appris pendant son pré-entraînement
    return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate)

In [ ]:
# Votre code ici

### Solution

In [ ]:
encoder_model = TransformerClassifier(ENCODER)
encoder_checkpoint = ModelCheckpoint(monitor="val_recall", mode="max")
encoder_trainer = lightning.Trainer(
    max_epochs=2,
    accelerator="auto",
    devices=1,
    # Calculs en demi-précision sur GPU : deux fois plus rapides
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    logger=CSVLogger("logs", name="distilroberta"),
    callbacks=[encoder_checkpoint])
encoder_trainer.fit(encoder_model, encoder_loaders["train"],
                    encoder_loaders["validation"])
encoder_trainer.test(encoder_model, encoder_loaders["test"], ckpt_path="best")

encoder_predictions = torch.cat(encoder_trainer.predict(
    encoder_model, encoder_loaders["test"], ckpt_path="best")).cpu().numpy()
show_evaluation(truths, encoder_predictions, "DistilRoBERTa affiné")
print("Rappel moyen sur les 1 000 premiers tweets de test")
for name, model_predictions in [("LSTM", predictions),
                                ("DistilRoBERTa affiné", encoder_predictions),
                                ("RoBERTa de cardiffnlp", roberta_predictions)]:
  score = recall_score(truths[:1000], model_predictions[:1000], average="macro")
  print(f"{name} : {score:.3f}")

En général, l'encodeur affiné en quelques minutes dépasse nettement le LSTM, et s'approche du RoBERTa de cardiffnlp sans l'égaler : ce dernier est plus gros et a été pré-entraîné sur des millions de tweets, dont il connaît le vocabulaire et les tournures. Affiner un encodeur pré-entraîné généraliste est la démarche standard quand aucun modèle n'existe déjà pour la tâche : ses représentations des mots et des phrases, apprises sur de grandes quantités de texte, suppléent au peu de données annotées.